In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("C:/Users/elisabethung/Downloads/Loan_Data.csv")

# =========================
# 2. DEFINE TARGET
# =========================
TARGET = "default"   # change if needed

# Drop obvious ID columns if present (VERY important)
if "customer_id" in df.columns:
    df = df.drop(columns=["customer_id"])

X = df.drop(columns=[TARGET])
y = df[TARGET]

# =========================
# 3. AUTO DETECT COLUMN TYPES
# =========================
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

# =========================
# 4. PREPROCESSING PIPELINE
# =========================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# =========================
# 5. MODELS
# =========================
log_reg = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42
    ))
])

# =========================
# 6. TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# 7. TRAIN MODELS
# =========================
log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)

# =========================
# 8. EVALUATION
# =========================
log_pred = log_reg.predict_proba(X_test)[:, 1]
rf_pred = rf.predict_proba(X_test)[:, 1]

print("Logistic Regression AUC:", roc_auc_score(y_test, log_pred))
print("Random Forest AUC:", roc_auc_score(y_test, rf_pred))

# Choose best model (usually RF)
best_model = rf

# =========================
# 9. EXPECTED LOSS FUNCTION
# =========================
RECOVERY_RATE = 0.10

def expected_loss(pd, exposure):
    lgd = 1 - RECOVERY_RATE
    return pd * lgd * exposure

# =========================
# 10. SAFE PREDICTION FUNCTION
# =========================
def predict_pd_and_el(borrower_dict, exposure):
    """
    borrower_dict: dict of borrower features
    exposure: loan amount (EAD)
    """

    input_df = pd.DataFrame([borrower_dict])

    # IMPORTANT: align columns automatically (fixes your error)
    input_df = input_df.reindex(columns=X.columns, fill_value=0)

    pd_value = best_model.predict_proba(input_df)[:, 1][0]
    el_value = expected_loss(pd_value, exposure)

    return {
        "probability_of_default": pd_value,
        "expected_loss": el_value
    }

# =========================
# 11. EXAMPLE USAGE
# =========================
sample_borrower = {
    # MUST match dataset columns (except target)
    col: 0 for col in X.columns   # safe dummy example
}

print(predict_pd_and_el(sample_borrower, exposure=20000))

Logistic Regression AUC: 0.9999883933012768
Random Forest AUC: 0.9998756425136792
{'probability_of_default': 0.17333333333333334, 'expected_loss': 3120.0}


In [7]:
for col in X.columns:
    print(col)

credit_lines_outstanding
loan_amt_outstanding
total_debt_outstanding
income
years_employed
fico_score


In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# -------------------
# LOAD DATA
# -------------------
df = pd.read_csv("C:/Users/elisabethung/Downloads/Loan_Data.csv")

TARGET = "default"  # adjust if needed

# Keep ONLY correct features
features = [
    "credit_lines_outstanding",
    "loan_amt_outstanding",
    "total_debt_outstanding",
    "income",
    "years_employed",
    "fico_score"
]

X = df[features]
y = df[TARGET]

# -------------------
# SPLIT
# -------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------
# MODELS
# -------------------
log_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42
    ))
])

# -------------------
# TRAIN
# -------------------
log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)

# -------------------
# EVALUATION
# -------------------
log_pred = log_reg.predict_proba(X_test)[:, 1]
rf_pred = rf.predict_proba(X_test)[:, 1]

print("LogReg AUC:", roc_auc_score(y_test, log_pred))
print("RF AUC:", roc_auc_score(y_test, rf_pred))

best_model = rf

# -------------------
# EXPECTED LOSS
# -------------------
RECOVERY_RATE = 0.10

def expected_loss(pd, exposure):
    return pd * (1 - RECOVERY_RATE) * exposure

# -------------------
# PREDICTION FUNCTION
# -------------------
def predict_pd_and_el(borrower_dict, exposure):
    input_df = pd.DataFrame([borrower_dict])
    input_df = input_df.reindex(columns=features, fill_value=0)

    pd_value = best_model.predict_proba(input_df)[:, 1][0]
    el_value = expected_loss(pd_value, exposure)

    return {
        "probability_of_default": pd_value,
        "expected_loss": el_value
    }

# -------------------
# EXAMPLE
# -------------------
sample_borrower = {
    "credit_lines_outstanding": 2,
    "loan_amt_outstanding": 15000,
    "total_debt_outstanding": 8000,
    "income": 60000,
    "years_employed": 5,
    "fico_score": 720
}

print(predict_pd_and_el(sample_borrower, exposure=15000))

LogReg AUC: 0.9999883933012768
RF AUC: 0.9998706682142264
{'probability_of_default': 0.0033333333333333335, 'expected_loss': 45.0}


In [11]:
corr = df.corr(numeric_only=True)["default"].sort_values(ascending=False)
print(corr)

default                     1.000000
credit_lines_outstanding    0.862815
total_debt_outstanding      0.758868
loan_amt_outstanding        0.098978
income                      0.016309
customer_id                 0.006927
years_employed             -0.284506
fico_score                 -0.324515
Name: default, dtype: float64


In [13]:
df.groupby("default").mean()


,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score
default,,,,,,,
0,4.967005e+06,0.744140,4092.629025,6322.164549,69883.889804,4.765247,646.938765
1,5.007914e+06,4.618044,4454.854897,19270.582961,70726.740140,3.617504,596.257699
